In [1]:
import random
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# CONFIGURATION
# =========================
TRAIN_SIZE = 18000
TEST_SIZE = 3000
FINAL_TEST_SIZE = 3000

# Output folders 
TRAIN_DIR = Path("../../../data/training")
TEST_DIR = Path("../../../data/test")
FINAL_TEST_DIR = Path("../../../data/final test")

for d in (TRAIN_DIR, TEST_DIR, FINAL_TEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["Temperature", "Pressure", "RPM", "Vibration"]

# =========================
# SAMPLE GENERATION (Critical Load)
# =========================
def gen_critical_load_sample():
    temp = random.uniform(115, 145)        # Overheating
    rpm  = random.uniform(4500, 9000)      # Near/above redline
    pressure  = 0.97 + random.uniform(-0.05, 0.05)
    vibration = 0.20 + (rpm - 5000.0) / 15000.0 + random.uniform(-0.08, 0.12)
    return [temp, pressure, rpm, vibration], "Critical Load"

def build_dataset(n):
    X, y, rows = [], [], []
    seq_counter = 1
    for _ in range(n):
        features, label = gen_critical_load_sample()
        X.append(features)
        y.append(label)
        rows.append({
            "Time": 1,
            "Sequence": seq_counter,
            "Temperature": features[0],
            "Pressure": features[1],
            "RPM": features[2],
            "Vibration": features[3],
            "State": label
        })
        seq_counter += 1
    return np.array(X, dtype=np.float32).reshape(n, 1, 4), np.array(y), pd.DataFrame(rows)

# =========================
# GENERATE & SAVE SPLITS
# =========================
# Training
X_tr, y_tr, df_tr = build_dataset(TRAIN_SIZE)
df_tr.to_csv(TRAIN_DIR / "CriticalLoad_training.csv", index=False)
np.save(TRAIN_DIR / "CriticalLoad_training_X.npy", X_tr)
np.save(TRAIN_DIR / "CriticalLoad_training_y.npy", y_tr)

# Test
X_te, y_te, df_te = build_dataset(TEST_SIZE)
df_te.to_csv(TEST_DIR / "CriticalLoad_test.csv", index=False)
np.save(TEST_DIR / "CriticalLoad_test_X.npy", X_te)
np.save(TEST_DIR / "CriticalLoad_test_y.npy", y_te)

# Final test
X_ft, y_ft, df_ft = build_dataset(FINAL_TEST_SIZE)
df_ft.to_csv(FINAL_TEST_DIR / "CriticalLoad_final test.csv", index=False)
np.save(FINAL_TEST_DIR / "CriticalLoad_final test_X.npy", X_ft)
np.save(FINAL_TEST_DIR / "CriticalLoad_final test_y.npy", y_ft)

print("✅ Critical Load datasets created.")
print("Training:", X_tr.shape, y_tr.shape)
print("Test:    ", X_te.shape, y_te.shape)
print("Final:   ", X_ft.shape, y_ft.shape)


✅ Critical Load datasets created.
Training: (18000, 1, 4) (18000,)
Test:     (3000, 1, 4) (3000,)
Final:    (3000, 1, 4) (3000,)
